In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [2]:
!nvidia-smi

Sat Sep 12 08:50:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os

In [4]:
import pandas as pd

BASE = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset"

nutrition = pd.read_csv(f"{BASE}/dish_nutrition_values.csv")

print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein']


,dish_id,calories,mass,fat,carb,protein
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970
1,dish_1562688426,137.569992,88.0,8.256000,5.190000,10.297000
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000


In [5]:
import os

nutrition["image_path"] = nutrition["dish_id"].apply(
    lambda x: f"{BASE}/imagery/realsense_overhead/{x}/rgb.png"
)

# Chỉ giữ món thực sự có ảnh RGB
nutrition = nutrition[
    nutrition["image_path"].apply(os.path.exists)
].reset_index(drop=True)

print("Số món có ảnh:", len(nutrition))

display(
    nutrition[
        ["dish_id", "image_path", "calories"]
    ].head()
)

Số món có ảnh: 3244


,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


In [6]:
print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein', 'image_path']


,dish_id,calories,mass,fat,carb,protein,image_path
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,/kaggle/input/datasets/gillesokhin/nutrition5k...
1,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,/kaggle/input/datasets/gillesokhin/nutrition5k...
2,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,/kaggle/input/datasets/gillesokhin/nutrition5k...
3,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,/kaggle/input/datasets/gillesokhin/nutrition5k...
4,dish_1558372433,74.360001,143.0,0.286000,0.429000,20.020000,/kaggle/input/datasets/gillesokhin/nutrition5k...


In [7]:
df = nutrition[
    ["dish_id", "image_path", "calories"]
].copy()

display(df.head())
print("Tổng số mẫu:", len(df))

,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


Tổng số mẫu: 3244


In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2270
Val: 487
Test: 487


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class NutritionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        calories = torch.tensor(row["calories"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, calories

In [10]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
train_dataset = NutritionDataset(train_df, train_transform)
val_dataset = NutritionDataset(val_df, val_transform)
test_dataset = NutritionDataset(test_df, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

2270 487 487


In [12]:
images, calories = next(iter(train_loader))

print("Images:", images.shape)
print("Calories:", calories.shape)
print("Ví dụ calories:", calories[:5])

Images: torch.Size([64, 3, 224, 224])
Calories: torch.Size([64])
Ví dụ calories: tensor([ 81.6600, 214.5278, 671.0820,  72.3608, 557.8840])


In [13]:
# Lưu split cố định để mọi experiment sau dùng cùng dữ liệu
train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)

print("Saved fixed splits:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Saved fixed splits:
Train: 2270
Validation: 487
Test: 487


In [14]:
# Thống kê mô tả target calories
def calorie_stats(dataframe, split_name):
    return {
        "split": split_name,
        "n_samples": len(dataframe),
        "min_kcal": dataframe["calories"].min(),
        "max_kcal": dataframe["calories"].max(),
        "mean_kcal": dataframe["calories"].mean(),
        "std_kcal": dataframe["calories"].std(),
        "median_kcal": dataframe["calories"].median(),
    }

stats_df = pd.DataFrame([
    calorie_stats(train_df, "Train"),
    calorie_stats(val_df, "Validation"),
    calorie_stats(test_df, "Test"),
    calorie_stats(pd.concat([train_df, val_df, test_df]), "Total"),
])

display(stats_df.round(2))
stats_df.to_csv("/kaggle/working/dataset_calorie_statistics.csv", index=False)


,split,n_samples,min_kcal,max_kcal,mean_kcal,std_kcal,median_kcal
0,Train,2270,0.00,3943.33,253.31,221.40,207.40
1,Validation,487,2.30,1324.08,256.67,212.09,212.16
2,Test,487,1.15,1264.55,266.35,218.93,234.58
3,Total,3244,0.00,3943.33,255.77,219.64,210.32


# EXPERIMENTS

Phần chuẩn bị dữ liệu phía trên được giữ cố định.  
Thêm các experiment mới từ đây trở xuống để bảo đảm tất cả thí nghiệm dùng cùng dataset split.


In [16]:
# ============================================================
# DEBUG STEP 1: KIỂM TRA CÁC TEST SAMPLE BỊ THIẾU DEPTH
# ============================================================

import os

test_debug = test_df.copy()

test_debug["depth_exists"] = (
    test_debug["depth_path"]
    .apply(os.path.exists)
)

missing_test = test_debug[
    ~test_debug["depth_exists"]
].copy()

print("Total test samples :", len(test_debug))
print("Missing depth      :", len(missing_test))

print("\n=== FIRST 5 MISSING SAMPLES ===")

for i, (_, row) in enumerate(
    missing_test.head(5).iterrows()
):
    print("\n----------------------------------")
    print("Sample:", i + 1)

    print(
        "RGB path:"
    )
    print(
        row["image_path"]
    )

    print(
        "\nDepth path đang tạo:"
    )
    print(
        row["depth_path"]
    )

    folder = os.path.dirname(
        row["image_path"]
    )

    print(
        "\nFolder:"
    )
    print(
        folder
    )

    print(
        "\nFiles trong folder:"
    )

    if os.path.exists(folder):

        files = os.listdir(folder)

        for f in files[:30]:
            print("  ", f)

    else:
        print(
            "Folder không tồn tại!"
        )

Total test samples : 487
Missing depth      : 1

=== FIRST 5 MISSING SAMPLES ===

----------------------------------
Sample: 1
RGB path:
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1558722871/rgb.png

Depth path đang tạo:
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1558722871/depth_raw.png

Folder:
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1558722871

Files trong folder:
   depth_raw.png_.gstmp
   rgb.png
   depth_color.png_.gstmp


In [17]:
# ============================================================
# DEBUG STEP 2: KIỂM TRA FILE .gstmp CÓ PHẢI DEPTH PNG HỢP LỆ?
# ============================================================

from PIL import Image
import os
import numpy as np

gstmp_path = (
    "/kaggle/input/datasets/gillesokhin/"
    "nutrition5k-dataset/imagery/realsense_overhead/"
    "dish_1558722871/depth_raw.png_.gstmp"
)

print("File exists:", os.path.exists(gstmp_path))
print("File size  :", os.path.getsize(gstmp_path), "bytes")

try:
    img = Image.open(gstmp_path)

    print("\n=== PIL READ SUCCESS ===")
    print("Format:", img.format)
    print("Mode  :", img.mode)
    print("Size  :", img.size)

    depth = np.array(img)

    print("Shape :", depth.shape)
    print("Dtype :", depth.dtype)
    print("Min   :", depth.min())
    print("Max   :", depth.max())

except Exception as e:
    print("\n=== PIL READ FAILED ===")
    print(type(e).__name__, ":", e)

File exists: True
File size  : 0 bytes

=== PIL READ FAILED ===
UnidentifiedImageError : cannot identify image file '/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1558722871/depth_raw.png_.gstmp'


In [18]:
# ============================================================
# STEP 3: TẠO FIXED SPLIT RIÊNG CHO RGB-D
# Chỉ loại sample thực sự không có depth
# KHÔNG thay đổi train_df / val_df / test_df gốc
# ============================================================

import os

train_df_rgbd = train_df.copy()
val_df_rgbd   = val_df.copy()
test_df_rgbd  = test_df.copy()


# ------------------------------------------------------------
# Giữ lại các sample có depth_raw.png hợp lệ
# ------------------------------------------------------------
train_df_rgbd = train_df_rgbd[
    train_df_rgbd["depth_path"].apply(os.path.exists)
].reset_index(drop=True)

val_df_rgbd = val_df_rgbd[
    val_df_rgbd["depth_path"].apply(os.path.exists)
].reset_index(drop=True)

test_df_rgbd = test_df_rgbd[
    test_df_rgbd["depth_path"].apply(os.path.exists)
].reset_index(drop=True)


# ------------------------------------------------------------
# KIỂM TRA
# ------------------------------------------------------------
print("=== ORIGINAL SPLIT ===")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("\n=== RGB-D VALID SPLIT ===")
print("Train:", len(train_df_rgbd))
print("Val  :", len(val_df_rgbd))
print("Test :", len(test_df_rgbd))

print(
    "\nTotal RGB-D:",
    len(train_df_rgbd)
    + len(val_df_rgbd)
    + len(test_df_rgbd)
)

print("\nAll depth valid?")

print(
    "Train:",
    train_df_rgbd["depth_path"]
    .apply(os.path.exists)
    .all()
)

print(
    "Val:",
    val_df_rgbd["depth_path"]
    .apply(os.path.exists)
    .all()
)

print(
    "Test:",
    test_df_rgbd["depth_path"]
    .apply(os.path.exists)
    .all()
)

=== ORIGINAL SPLIT ===
Train: 2270
Val  : 487
Test : 487

=== RGB-D VALID SPLIT ===
Train: 2270
Val  : 487
Test : 486

Total RGB-D: 3243

All depth valid?
Train: True
Val: True
Test: True


In [19]:
# ============================================================
# STEP 4: TÍNH DEPTH NORMALIZATION TỪ TRAIN SET
# Chỉ sử dụng train_df_rgbd
# ============================================================

from PIL import Image
import numpy as np

rng = np.random.default_rng(42)

sampled_depth_values = []

for path in train_df_rgbd["depth_path"]:

    depth = np.array(
        Image.open(path),
        dtype=np.float32
    )

    # Chỉ lấy pixel depth hợp lệ
    valid = depth[depth > 0]

    if len(valid) == 0:
        continue

    # Lấy tối đa 1000 pixel / ảnh
    n_sample = min(
        1000,
        len(valid)
    )

    idx = rng.choice(
        len(valid),
        size=n_sample,
        replace=False
    )

    sampled_depth_values.append(
        valid[idx]
    )


sampled_depth_values = np.concatenate(
    sampled_depth_values
)


DEPTH_LOW = np.percentile(
    sampled_depth_values,
    1
)

DEPTH_HIGH = np.percentile(
    sampled_depth_values,
    99
)


print("=== DEPTH NORMALIZATION FROM TRAIN ===")
print("Number of sampled pixels:", len(sampled_depth_values))
print("P1  :", DEPTH_LOW)
print("P99 :", DEPTH_HIGH)

print(
    "\nValid range:",
    DEPTH_HIGH > DEPTH_LOW
)

=== DEPTH NORMALIZATION FROM TRAIN ===
Number of sampled pixels: 2270000
P1  : 3038.0
P99 : 4860.0

Valid range: True


In [20]:
# ============================================================
# STEP 5: TẠO RGB-D DATASET + KIỂM TRA 1 SAMPLE
# ============================================================

import random
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import functional as TF
from PIL import Image
import numpy as np


# ------------------------------------------------------------
# RGB NORMALIZATION
# ------------------------------------------------------------
RGB_NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)


class NutritionRGBDDataset(Dataset):

    def __init__(
        self,
        dataframe,
        depth_low,
        depth_high,
        train=True
    ):
        self.df = dataframe.reset_index(drop=True)
        self.depth_low = depth_low
        self.depth_high = depth_high
        self.train = train


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ====================================================
        # 1. RGB
        # ====================================================
        rgb = Image.open(
            row["image_path"]
        ).convert("RGB")


        # ====================================================
        # 2. DEPTH
        # ====================================================
        depth = np.array(
            Image.open(row["depth_path"]),
            dtype=np.float32
        )

        valid_mask = depth > 0

        depth_norm = np.zeros_like(
            depth,
            dtype=np.float32
        )

        # chỉ normalize pixel depth hợp lệ
        clipped = np.clip(
            depth[valid_mask],
            self.depth_low,
            self.depth_high
        )

        depth_norm[valid_mask] = (
            clipped - self.depth_low
        ) / (
            self.depth_high - self.depth_low
        )

        depth_tensor = torch.from_numpy(
            depth_norm
        ).unsqueeze(0)   # [1, H, W]


        # ====================================================
        # 3. RESIZE CÙNG KÍCH THƯỚC
        # ====================================================
        rgb = TF.resize(
            rgb,
            [224, 224]
        )

        depth_tensor = TF.resize(
            depth_tensor,
            [224, 224],
            antialias=True
        )


        # ====================================================
        # 4. AUGMENTATION ĐỒNG BỘ RGB + DEPTH
        # ====================================================
        if self.train and random.random() < 0.5:

            rgb = TF.hflip(rgb)
            depth_tensor = TF.hflip(
                depth_tensor
            )


        # ====================================================
        # 5. RGB -> TENSOR + NORMALIZE
        # ====================================================
        rgb = TF.to_tensor(rgb)

        rgb = RGB_NORMALIZE(rgb)


        # ====================================================
        # 6. TARGET
        # ====================================================
        calories = torch.tensor(
            row["calories"],
            dtype=torch.float32
        )


        return (
            rgb,
            depth_tensor,
            calories
        )


# ============================================================
# CREATE DATASETS
# ============================================================

train_rgbd_dataset = NutritionRGBDDataset(
    train_df_rgbd,
    DEPTH_LOW,
    DEPTH_HIGH,
    train=True
)

val_rgbd_dataset = NutritionRGBDDataset(
    val_df_rgbd,
    DEPTH_LOW,
    DEPTH_HIGH,
    train=False
)

test_rgbd_dataset = NutritionRGBDDataset(
    test_df_rgbd,
    DEPTH_LOW,
    DEPTH_HIGH,
    train=False
)


# ============================================================
# SANITY CHECK 1 SAMPLE
# ============================================================

rgb, depth, calories = train_rgbd_dataset[0]

print("=== DATASET CHECK ===")

print("Train samples:", len(train_rgbd_dataset))
print("Val samples  :", len(val_rgbd_dataset))
print("Test samples :", len(test_rgbd_dataset))

print("\nRGB shape   :", rgb.shape)
print("Depth shape :", depth.shape)
print("Calories    :", calories.item())

print(
    "\nRGB min/max:",
    rgb.min().item(),
    rgb.max().item()
)

print(
    "Depth min/max:",
    depth.min().item(),
    depth.max().item()
)

print(
    "Depth NaN:",
    torch.isnan(depth).any().item()
)

print(
    "RGB NaN:",
    torch.isnan(rgb).any().item()
)

=== DATASET CHECK ===
Train samples: 2270
Val samples  : 487
Test samples : 486

RGB shape   : torch.Size([3, 224, 224])
Depth shape : torch.Size([1, 224, 224])
Calories    : 351.405517578125

RGB min/max: -2.1179039478302 1.472244143486023
Depth min/max: 0.0 1.000000238418579
Depth NaN: False
RGB NaN: False


In [21]:
# ============================================================
# STEP 6: CREATE DATALOADERS + CHECK ONE BATCH
# ============================================================

from torch.utils.data import DataLoader

BATCH_SIZE = 64


train_rgbd_loader = DataLoader(
    train_rgbd_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_rgbd_loader = DataLoader(
    val_rgbd_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_rgbd_loader = DataLoader(
    test_rgbd_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# CHECK ONE BATCH
# ============================================================

rgb_batch, depth_batch, calories_batch = next(
    iter(train_rgbd_loader)
)

print("=== DATALOADER CHECK ===")

print("RGB batch shape     :", rgb_batch.shape)
print("Depth batch shape   :", depth_batch.shape)
print("Calories batch shape:", calories_batch.shape)

print("\nRGB dtype   :", rgb_batch.dtype)
print("Depth dtype :", depth_batch.dtype)
print("Target dtype:", calories_batch.dtype)

print(
    "\nDepth min/max:",
    depth_batch.min().item(),
    depth_batch.max().item()
)

print(
    "RGB NaN:",
    torch.isnan(rgb_batch).any().item()
)

print(
    "Depth NaN:",
    torch.isnan(depth_batch).any().item()
)

print(
    "Target NaN:",
    torch.isnan(calories_batch).any().item()
)

print("\nNumber of batches:")
print("Train:", len(train_rgbd_loader))
print("Val  :", len(val_rgbd_loader))
print("Test :", len(test_rgbd_loader))

=== DATALOADER CHECK ===
RGB batch shape     : torch.Size([64, 3, 224, 224])
Depth batch shape   : torch.Size([64, 1, 224, 224])
Calories batch shape: torch.Size([64])

RGB dtype   : torch.float32
Depth dtype : torch.float32
Target dtype: torch.float32

Depth min/max: 0.0 1.000000238418579
RGB NaN: False
Depth NaN: False
Target NaN: False

Number of batches:
Train: 36
Val  : 8
Test : 8


In [22]:
# ============================================================
# STEP 7: TẠO RGB-D BRANCH MODEL PARALLEL
#
# GPU 0: RGB   -> ResNet50
# GPU 1: Depth -> ResNet18
# GPU 1: Concat + Fusion -> Calories
#
# CHƯA TRAIN
# ============================================================

import gc
import torch
import torch.nn as nn

from torchvision.models import (
    resnet18,
    resnet50,
    ResNet18_Weights,
    ResNet50_Weights
)


# ============================================================
# MODEL
# ============================================================

class RGBDBranchModelParallel(nn.Module):

    def __init__(
        self,
        rgb_device="cuda:0",
        depth_device="cuda:1"
    ):
        super().__init__()

        self.rgb_device = torch.device(rgb_device)
        self.depth_device = torch.device(depth_device)


        # ====================================================
        # GPU 0: RGB -> ResNet50
        # ====================================================
        self.rgb_encoder = resnet50(
            weights=ResNet50_Weights.DEFAULT
        )

        rgb_dim = self.rgb_encoder.fc.in_features

        self.rgb_encoder.fc = nn.Identity()

        self.rgb_encoder = self.rgb_encoder.to(
            self.rgb_device
        )


        # ====================================================
        # GPU 1: Depth -> ResNet18
        # ====================================================
        self.depth_encoder = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        # ResNet18 mặc định nhận 3 channel
        # Depth chỉ có 1 channel
        old_conv = self.depth_encoder.conv1

        new_conv = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )

        # Khởi tạo weight cho depth bằng trung bình
        # 3 channel RGB pretrained
        with torch.no_grad():

            new_conv.weight.copy_(
                old_conv.weight.mean(
                    dim=1,
                    keepdim=True
                )
            )

        self.depth_encoder.conv1 = new_conv

        depth_dim = self.depth_encoder.fc.in_features

        self.depth_encoder.fc = nn.Identity()

        self.depth_encoder = self.depth_encoder.to(
            self.depth_device
        )


        # ====================================================
        # GPU 1: FUSION HEAD
        # ====================================================
        self.fusion = nn.Sequential(

            nn.Linear(
                rgb_dim + depth_dim,   # 2048 + 512
                512
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                512,
                1
            )

        ).to(
            self.depth_device
        )


    def forward(
        self,
        rgb,
        depth
    ):

        # GPU 0
        rgb_feature = self.rgb_encoder(
            rgb
        )
        # shape: [B, 2048]


        # GPU 1
        depth_feature = self.depth_encoder(
            depth
        )
        # shape: [B, 512]


        # ====================================================
        # TRANSFER FEATURE GPU 0 -> GPU 1
        # ====================================================
        rgb_feature = rgb_feature.to(
            self.depth_device,
            non_blocking=True
        )


        # GPU 1
        fused_feature = torch.cat(
            [
                rgb_feature,
                depth_feature
            ],
            dim=1
        )
        # shape: [B, 2560]


        output = self.fusion(
            fused_feature
        )

        return output


# ============================================================
# CLEAN GPU
# ============================================================

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# CREATE MODEL
# ============================================================

model_mp = RGBDBranchModelParallel(
    rgb_device="cuda:0",
    depth_device="cuda:1"
)


# ============================================================
# CHECK DEVICE PLACEMENT
# ============================================================

print("=== MODEL DEVICE CHECK ===")

print(
    "RGB encoder  :",
    next(model_mp.rgb_encoder.parameters()).device
)

print(
    "Depth encoder:",
    next(model_mp.depth_encoder.parameters()).device
)

print(
    "Fusion head  :",
    next(model_mp.fusion.parameters()).device
)


# ============================================================
# GET ONE BATCH
# ============================================================

rgb_batch, depth_batch, calories_batch = next(
    iter(train_rgbd_loader)
)


# Toàn bộ RGB batch -> GPU 0
rgb_batch = rgb_batch.to(
    "cuda:0"
)

# Toàn bộ Depth batch -> GPU 1
depth_batch = depth_batch.to(
    "cuda:1"
)


print("\n=== INPUT DEVICE CHECK ===")

print(
    "RGB batch device  :",
    rgb_batch.device
)

print(
    "Depth batch device:",
    depth_batch.device
)


# ============================================================
# TEST FORWARD
# ============================================================

model_mp.eval()

with torch.no_grad():

    output = model_mp(
        rgb_batch,
        depth_batch
    )


print("\n=== FORWARD CHECK ===")

print(
    "Output shape :",
    output.shape
)

print(
    "Output device:",
    output.device
)

print(
    "Output NaN   :",
    torch.isnan(output).any().item()
)


# ============================================================
# GPU MEMORY CHECK
# ============================================================

print("\n=== GPU MEMORY ===")

print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        2
    ),
    "GB"
)

print(
    "GPU 1 allocated:",
    round(
        torch.cuda.memory_allocated(1)
        / 1024**3,
        2
    ),
    "GB"
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s] 


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 157MB/s] 


=== MODEL DEVICE CHECK ===
RGB encoder  : cuda:0
Depth encoder: cuda:1
Fusion head  : cuda:1

=== INPUT DEVICE CHECK ===
RGB batch device  : cuda:0
Depth batch device: cuda:1

=== FORWARD CHECK ===
Output shape : torch.Size([64, 1])
Output device: cuda:1
Output NaN   : False

=== GPU MEMORY ===
GPU 0 allocated: 0.12 GB
GPU 1 allocated: 0.07 GB


In [23]:
# ============================================================
# STEP 8: BACKWARD / GRADIENT CHECK
# ============================================================

model_mp.train()

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model_mp.parameters(),
    lr=1e-4,
    foreach=False
)

# Lấy 1 batch mới
rgb_batch, depth_batch, calories_batch = next(
    iter(train_rgbd_loader)
)

# RGB -> GPU 0
rgb_batch = rgb_batch.to(
    model_mp.rgb_device
)

# Depth + target -> GPU 1
depth_batch = depth_batch.to(
    model_mp.depth_device
)

calories_batch = calories_batch.to(
    model_mp.depth_device
)

optimizer.zero_grad(
    set_to_none=True
)

# Forward
output = model_mp(
    rgb_batch,
    depth_batch
).squeeze(1)

loss = criterion(
    output,
    calories_batch
)

print("Loss before backward:", loss.item())

# Backward
loss.backward()


# ============================================================
# CHECK GRADIENTS
# ============================================================

rgb_grad = (
    model_mp.rgb_encoder.conv1.weight.grad
)

depth_grad = (
    model_mp.depth_encoder.conv1.weight.grad
)

fusion_grad = (
    model_mp.fusion[0].weight.grad
)


print("\n=== GRADIENT CHECK ===")

print(
    "RGB branch grad exists:",
    rgb_grad is not None
)

print(
    "Depth branch grad exists:",
    depth_grad is not None
)

print(
    "Fusion grad exists:",
    fusion_grad is not None
)


if rgb_grad is not None:
    print(
        "RGB grad mean abs:",
        rgb_grad.abs().mean().item()
    )

if depth_grad is not None:
    print(
        "Depth grad mean abs:",
        depth_grad.abs().mean().item()
    )

if fusion_grad is not None:
    print(
        "Fusion grad mean abs:",
        fusion_grad.abs().mean().item()
    )


# ============================================================
# TEST 1 OPTIMIZER STEP
# ============================================================

optimizer.step()

print("\nOptimizer step: SUCCESS")

Loss before backward: 287.2803649902344

=== GRADIENT CHECK ===
RGB branch grad exists: True
Depth branch grad exists: True
Fusion grad exists: True
RGB grad mean abs: 0.0019476368324831128
Depth grad mean abs: 0.012319032102823257
Fusion grad mean abs: 0.003353595035150647

Optimizer step: SUCCESS


In [25]:
# ============================================================
# DEBUG STEP: QUÉT TOÀN BỘ DEPTH FILE BỊ HỎNG / KHÔNG ĐỌC ĐƯỢC
# ============================================================

from PIL import Image
import os


def check_depth_files(df, name):

    broken = []

    for idx, row in df.iterrows():

        path = row["depth_path"]

        # File không tồn tại
        if not os.path.exists(path):
            broken.append({
                "index": idx,
                "dish_id": row.get("dish_id", None),
                "path": path,
                "reason": "missing"
            })
            continue

        # File tồn tại nhưng thử đọc
        try:
            with Image.open(path) as img:
                img.verify()

        except Exception as e:
            broken.append({
                "index": idx,
                "dish_id": row.get("dish_id", None),
                "path": path,
                "reason": f"{type(e).__name__}: {e}"
            })


    print(f"\n=== {name} ===")
    print("Total samples :", len(df))
    print("Broken depth  :", len(broken))

    for item in broken[:10]:
        print("\n---")
        print("index :", item["index"])
        print("dish  :", item["dish_id"])
        print("path  :", item["path"])
        print("error :", item["reason"])

    return broken


broken_train = check_depth_files(
    train_df_rgbd,
    "TRAIN"
)

broken_val = check_depth_files(
    val_df_rgbd,
    "VALIDATION"
)

broken_test = check_depth_files(
    test_df_rgbd,
    "TEST"
)


print("\n==============================")
print("SUMMARY")
print("==============================")

print("Train broken:", len(broken_train))
print("Val broken  :", len(broken_val))
print("Test broken :", len(broken_test))


=== TRAIN ===
Total samples : 2270
Broken depth  : 0

=== VALIDATION ===
Total samples : 487
Broken depth  : 1

---
index : 233
dish  : dish_1564159636
path  : /kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1564159636/depth_raw.png
error : UnidentifiedImageError: cannot identify image file '/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1564159636/depth_raw.png'

=== TEST ===
Total samples : 486
Broken depth  : 0

SUMMARY
Train broken: 0
Val broken  : 1
Test broken : 0


In [26]:
# ============================================================
# STEP 10: REMOVE 1 BROKEN DEPTH SAMPLE FROM VALIDATION
# ============================================================

BROKEN_DISH_ID = "dish_1564159636"

print("Before:")
print("Train:", len(train_df_rgbd))
print("Val  :", len(val_df_rgbd))
print("Test :", len(test_df_rgbd))


# ------------------------------------------------------------
# REMOVE ONLY THE BROKEN VALIDATION SAMPLE
# ------------------------------------------------------------
val_df_rgbd = val_df_rgbd[
    val_df_rgbd["dish_id"] != BROKEN_DISH_ID
].reset_index(drop=True)


print("\nAfter:")
print("Train:", len(train_df_rgbd))
print("Val  :", len(val_df_rgbd))
print("Test :", len(test_df_rgbd))


# ------------------------------------------------------------
# RECREATE VALIDATION DATASET
# ------------------------------------------------------------
val_rgbd_dataset = NutritionRGBDDataset(
    val_df_rgbd,
    DEPTH_LOW,
    DEPTH_HIGH,
    train=False
)


# ------------------------------------------------------------
# RECREATE VALIDATION DATALOADER
# ------------------------------------------------------------
val_rgbd_loader = DataLoader(
    val_rgbd_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# CHECK ALL VALIDATION SAMPLES CAN ACTUALLY LOAD
# ------------------------------------------------------------
count = 0

for rgb, depth, calories in val_rgbd_loader:
    count += len(rgb)


print("\n=== VALIDATION CHECK ===")
print("Loaded samples:", count)
print("Dataset size  :", len(val_rgbd_dataset))
print("Batches       :", len(val_rgbd_loader))
print("SUCCESS       :", count == len(val_rgbd_dataset))

Before:
Train: 2270
Val  : 487
Test : 486

After:
Train: 2270
Val  : 486
Test : 486

=== VALIDATION CHECK ===
Loaded samples: 486
Dataset size  : 486
Batches       : 8
SUCCESS       : True


In [27]:
# ============================================================
# STEP 11: RERUN CLEAN 1-EPOCH SMOKE TEST
# ============================================================

import gc
import time
import random
import numpy as np
import torch
import torch.nn as nn


# ------------------------------------------------------------
# CLEAN OLD MODEL
# ------------------------------------------------------------
if "model_mp" in globals():
    del model_mp

if "optimizer" in globals():
    del optimizer

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# FIX SEED
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# CREATE COMPLETELY FRESH MODEL
# ------------------------------------------------------------
model_mp = RGBDBranchModelParallel(
    rgb_device="cuda:0",
    depth_device="cuda:1"
)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model_mp.parameters(),
    lr=1e-4,
    foreach=False
)


# ------------------------------------------------------------
# RESET VRAM STATS
# ------------------------------------------------------------
torch.cuda.reset_peak_memory_stats(0)
torch.cuda.reset_peak_memory_stats(1)


# ------------------------------------------------------------
# TRAIN 1 EPOCH
# ------------------------------------------------------------
torch.cuda.synchronize(0)
torch.cuda.synchronize(1)

start_time = time.time()

train_loss = train_one_epoch_mp(
    model_mp,
    train_rgbd_loader,
    criterion,
    optimizer
)

torch.cuda.synchronize(0)
torch.cuda.synchronize(1)

train_time = time.time() - start_time


# ------------------------------------------------------------
# VALIDATION
# Không tính vào training time
# ------------------------------------------------------------
val_loss, val_mae, val_rmse = evaluate_mp(
    model_mp,
    val_rgbd_loader,
    criterion
)


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------
throughput = (
    len(train_rgbd_dataset)
    / train_time
)

gpu0_peak = (
    torch.cuda.max_memory_allocated(0)
    / 1024**3
)

gpu1_peak = (
    torch.cuda.max_memory_allocated(1)
    / 1024**3
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------
print("=== CLEAN 1-EPOCH MODEL PARALLEL TEST ===")

print(f"Train Loss : {train_loss:.3f}")
print(f"Val Loss   : {val_loss:.3f}")

print(f"Val MAE    : {val_mae:.2f} kcal")
print(f"Val RMSE   : {val_rmse:.2f} kcal")

print(f"Train Time : {train_time:.2f} s")
print(f"Throughput : {throughput:.2f} samples/s")

print(f"GPU 0 Peak : {gpu0_peak:.2f} GB")
print(f"GPU 1 Peak : {gpu1_peak:.2f} GB")

=== CLEAN 1-EPOCH MODEL PARALLEL TEST ===
Train Loss : 235.969
Val Loss   : 151.292
Val MAE    : 151.79 kcal
Val RMSE   : 222.64 kcal
Train Time : 41.27 s
Throughput : 55.00 samples/s
GPU 0 Peak : 5.89 GB
GPU 1 Peak : 1.85 GB


In [28]:
# ============================================================
# STEP 12: OFFICIAL 20-EPOCH MODEL PARALLEL EXPERIMENT
#
# GPU 0: RGB   -> ResNet50
# GPU 1: Depth -> ResNet18
# GPU 1: Fusion -> Calories
#
# Training time KHÔNG bao gồm validation
# ============================================================

import gc
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# ============================================================
# CONFIG
# ============================================================

SEED = 42
EPOCHS = 20
LR = 1e-4

BEST_MODEL_PATH = (
    "/kaggle/working/"
    "rgbd_branch_model_parallel_best.pth"
)

HISTORY_PATH = (
    "/kaggle/working/"
    "rgbd_branch_model_parallel_history.csv"
)


# ============================================================
# CLEAN OLD EXPERIMENT
# ============================================================

if "model_mp" in globals():
    del model_mp

if "optimizer" in globals():
    del optimizer

gc.collect()

torch.cuda.empty_cache()


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# COMPLETELY FRESH MODEL
# ============================================================

model_mp = RGBDBranchModelParallel(
    rgb_device="cuda:0",
    depth_device="cuda:1"
)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model_mp.parameters(),
    lr=LR,
    foreach=False
)


# ============================================================
# VARIABLES
# ============================================================

history = []

best_mae = float("inf")
best_rmse = None
best_epoch = None

total_train_time = 0.0

max_gpu0_vram = 0.0
max_gpu1_vram = 0.0


# ============================================================
# TRAIN 20 EPOCHS
# ============================================================

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # Reset VRAM statistics for this epoch
    # --------------------------------------------------------

    torch.cuda.reset_peak_memory_stats(0)
    torch.cuda.reset_peak_memory_stats(1)


    # --------------------------------------------------------
    # TRAINING TIMER
    # --------------------------------------------------------

    torch.cuda.synchronize(0)
    torch.cuda.synchronize(1)

    start_time = time.time()


    train_loss = train_one_epoch_mp(
        model_mp,
        train_rgbd_loader,
        criterion,
        optimizer
    )


    torch.cuda.synchronize(0)
    torch.cuda.synchronize(1)

    epoch_train_time = (
        time.time() - start_time
    )

    total_train_time += epoch_train_time


    # --------------------------------------------------------
    # Training throughput
    # --------------------------------------------------------

    epoch_throughput = (
        len(train_rgbd_dataset)
        / epoch_train_time
    )


    # --------------------------------------------------------
    # VRAM
    # --------------------------------------------------------

    gpu0_vram = (
        torch.cuda.max_memory_allocated(0)
        / 1024**3
    )

    gpu1_vram = (
        torch.cuda.max_memory_allocated(1)
        / 1024**3
    )

    max_gpu0_vram = max(
        max_gpu0_vram,
        gpu0_vram
    )

    max_gpu1_vram = max(
        max_gpu1_vram,
        gpu1_vram
    )


    # --------------------------------------------------------
    # VALIDATION
    # NOT included in training time
    # --------------------------------------------------------

    (
        val_loss,
        val_mae,
        val_rmse
    ) = evaluate_mp(
        model_mp,
        val_rgbd_loader,
        criterion
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL BY VAL MAE
    # --------------------------------------------------------

    if val_mae < best_mae:

        best_mae = val_mae
        best_rmse = val_rmse
        best_epoch = epoch + 1

        torch.save(
            model_mp.state_dict(),
            BEST_MODEL_PATH
        )


    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    history.append({

        "epoch":
            epoch + 1,

        "train_loss":
            train_loss,

        "val_loss":
            val_loss,

        "val_mae":
            val_mae,

        "val_rmse":
            val_rmse,

        "train_time_sec":
            epoch_train_time,

        "throughput_samples_sec":
            epoch_throughput,

        "gpu0_vram_gb":
            gpu0_vram,

        "gpu1_vram_gb":
            gpu1_vram
    })


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.3f} | "
        f"Val MAE: {val_mae:.2f} | "
        f"Val RMSE: {val_rmse:.2f} | "
        f"Time: {epoch_train_time:.2f}s | "
        f"Throughput: {epoch_throughput:.2f} | "
        f"GPU0: {gpu0_vram:.2f}GB | "
        f"GPU1: {gpu1_vram:.2f}GB"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

history_df = pd.DataFrame(history)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)


overall_throughput = (
    len(train_rgbd_dataset)
    * EPOCHS
    / total_train_time
)


print("\n")
print("=" * 60)

print(
    "RGB-D BRANCH MODEL PARALLEL - 20 EPOCH RESULT"
)

print("=" * 60)

print(
    f"Total training time : "
    f"{total_train_time:.2f} s"
)

print(
    f"Overall throughput  : "
    f"{overall_throughput:.2f} samples/s"
)

print(
    f"Best epoch          : "
    f"{best_epoch}"
)

print(
    f"Best Val MAE        : "
    f"{best_mae:.2f} kcal"
)

print(
    f"Best Val RMSE       : "
    f"{best_rmse:.2f} kcal"
)

print(
    f"Peak GPU 0 VRAM     : "
    f"{max_gpu0_vram:.2f} GB"
)

print(
    f"Peak GPU 1 VRAM     : "
    f"{max_gpu1_vram:.2f} GB"
)

print("=" * 60)

Epoch 01/20 | Train Loss: 235.983 | Val MAE: 154.06 | Val RMSE: 225.43 | Time: 40.12s | Throughput: 56.58 | GPU0: 5.89GB | GPU1: 1.85GB
Epoch 02/20 | Train Loss: 133.213 | Val MAE: 80.14 | Val RMSE: 111.11 | Time: 38.84s | Throughput: 58.44 | GPU0: 5.89GB | GPU1: 1.85GB
Epoch 03/20 | Train Loss: 70.320 | Val MAE: 68.69 | Val RMSE: 106.53 | Time: 39.46s | Throughput: 57.53 | GPU0: 5.89GB | GPU1: 1.85GB
Epoch 04/20 | Train Loss: 60.811 | Val MAE: 55.32 | Val RMSE: 88.53 | Time: 38.93s | Throughput: 58.32 | GPU0: 5.89GB | GPU1: 1.85GB
Epoch 05/20 | Train Loss: 49.404 | Val MAE: 52.35 | Val RMSE: 81.85 | Time: 39.59s | Throughput: 57.34 | GPU0: 5.89GB | GPU1: 1.85GB
Epoch 06/20 | Train Loss: 47.405 | Val MAE: 49.63 | Val RMSE: 76.00 | Time: 39.74s | Throughput: 57.13 | GPU0: 5.89GB | GPU1: 1.85GB
Epoch 07/20 | Train Loss: 42.172 | Val MAE: 48.17 | Val RMSE: 79.74 | Time: 40.51s | Throughput: 56.04 | GPU0: 5.89GB | GPU1: 1.85GB
Epoch 08/20 | Train Loss: 41.219 | Val MAE: 50.28 | Val RMSE: 8